# Week 89, Day 2: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('waymark_geo.py', 'w', encoding='utf-8') as f:
    f.write('import math\nfrom waymark_domain import PathNetwork\nfrom waymark_map import HEIGHTS, PATHS\n\nPOSITIONS = {"Mill": (0.0, 0.0), "Weir": (1.4, 0.4), "Church": (2.8, 1.2), "Ridge": (4.4, 3.4), "Farm": (2.8, 2.4),\n             "Wood": (2.0, 3.5), "Lake": (0.6, 2.8), "Island": (-3.0, 5.0)}\n\n\nclass GeoNetwork(PathNetwork):\n    """A path network whose junctions also have positions, in kilometres east and north."""\n\n    def __init__(self, heights, positions):\n        super().__init__(heights)\n        self.positions = dict(positions)\n\n    def straight_km(self, a, b):\n        return math.dist(self.positions[a], self.positions[b])\n\n\ndef sample_geo():\n    network = GeoNetwork(HEIGHTS, POSITIONS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n\n\ndef grid_network(size):\n    """A size-by-size grid of junctions named \'r-c\', 1 km apart, with slightly longer paths to their neighbours."""\n    names = {(r, c): f"{r}-{c}" for r in range(size) for c in range(size)}\n    heights = {name: ((r * 13 + c * 7) % 9) * 20 for (r, c), name in names.items()}\n    positions = {name: (float(c), float(r)) for (r, c), name in names.items()}\n    network = GeoNetwork(heights, positions)\n    for (r, c), name in names.items():\n        if c + 1 < size:\n            network.add_path(name, names[(r, c + 1)], 1.0 + ((r * 7 + c * 3) % 5) / 10)\n        if r + 1 < size:\n            network.add_path(name, names[(r + 1, c)], 1.0 + ((r * 3 + c * 5) % 5) / 10)\n    return network\n')

**Predict before you run:** what will this print?



In [ ]:
from waymark_geo import grid_network

def count_loops(network, start, max_stops):
    count = 0
    def explore(path):
        nonlocal count
        for nxt in network.paths[path[-1]]:
            if nxt == start and len(path) >= 3:
                count += 1
            elif nxt not in path and len(path) < max_stops:
                explore(path + [nxt])
    explore([start])
    return count

grid = grid_network(5)
print([count_loops(grid, "2-2", n) for n in (4, 6, 8, 10)])

> **Key idea.** Exhaustive searches that grow exponentially need a different idea, not a faster machine. Dijkstra run to completion gives distances and a shortest-path tree from one start to everywhere; follow `prev` back to rebuild any route. Search while avoiding paths by skipping banned pairs (both directions), never by changing a shared network. A round walk is out to a turning point and back by the shortest route avoiding the outward paths; choose turning points at 30% to 60% of the target distance, skip ones with no way back, keep loops within the time tolerance, and deduplicate by the set of paths used.

### Exercise: Distances to everywhere

Write `shortest_tree(network, start)`: run Dijkstra's algorithm from `start` until the heap is empty (neighbours in name order). Return `(dist, prev)`: `dist` maps every reachable junction (including `start`, at 0) to its distance rounded to 2 places; `prev` maps every reachable junction except `start` to the junction before it on its shortest route. Then `route_to(prev, start, end)`: the list of stops from `start` to `end`, or `None` if `end` isn't reachable.

In [ ]:
import heapq
from waymark_geo import sample_geo

def shortest_tree(network, start):
    ___

def route_to(prev, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo\n\ndef test_tree():\n    \"one search, every answer\"\n    net = sample_geo()\n    dist, prev = shortest_tree(net, \"Mill\")\n    assert dist == {\"Mill\": 0, \"Weir\": 1.5, \"Church\": 3.5, \"Farm\": 4.0, \"Lake\": 3.5, \"Ridge\": 6.0, \"Wood\": 5.5}\n    assert route_to(prev, \"Mill\", \"Ridge\") == [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"] and route_to(prev, \"Mill\", \"Mill\") == [\"Mill\"]\n    assert route_to(prev, \"Mill\", \"Island\") is None and \"Island\" not in dist\n    for end in dist:\n        assert dist[end] == net.shortest(\"Mill\", end).km")

### Exercise: Searching around banned paths

Write `shortest_avoiding(network, start, end, banned)`: the shortest route as a domain `Route` (km rounded to 2 places, climb from the network) that never uses a path in `banned` (a set of `frozenset({a, b})` pairs, banned in both directions), or `None` if there's no such route. Don't change the network. Then `paths_of(stops)`: the set of `frozenset` pairs a list of stops uses.

In [ ]:
import heapq
from waymark_domain import Route
from waymark_geo import sample_geo

def shortest_avoiding(network, start, end, banned):
    ___

def paths_of(stops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo\n\ndef test_banned():\n    \"around, not along\"\n    net = sample_geo()\n    before = {k: dict(v) for k, v in net.paths.items()}\n    out = [\"Mill\", \"Weir\", \"Farm\"]\n    back = shortest_avoiding(net, \"Farm\", \"Mill\", paths_of(out))\n    assert back.stops == [\"Farm\", \"Wood\", \"Lake\", \"Mill\"] and back.km == 7.0 and back.climb_m == 30\n    assert net.paths == before, \"the network is unchanged\"\n    assert paths_of([\"A\", \"B\", \"C\"]) == {frozenset({\"A\", \"B\"}), frozenset({\"B\", \"C\"})}\n    assert shortest_avoiding(net, \"Mill\", \"Weir\", set()).km == 1.5\n    assert shortest_avoiding(net, \"Mill\", \"Island\", set()) is None\n    cut = {frozenset({\"Mill\", \"Weir\"}), frozenset({\"Mill\", \"Church\"}), frozenset({\"Mill\", \"Lake\"})}\n    assert shortest_avoiding(net, \"Mill\", \"Weir\", cut) is None")

### Exercise: A loop through a turning point

Using the given helpers, write `loop_via(network, start, turn)`: the outward route is the shortest from `start` to `turn`; the return is `shortest_avoiding` from `turn` back to `start` with the outward paths banned. Return a `Route` for the whole loop (stops joined without repeating the turning point, km the sum rounded to 2 places, climb over all the stops), or `None` if there's no way back, or if `turn` is `start`.

In [ ]:
import heapq
from waymark_domain import Route
from waymark_geo import sample_geo

def shortest_avoiding(network, start, end, banned):
    dist, prev, done, heap = {start: 0}, {}, set(), [(0, start)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        if node == end:
            stops = [end]
            while stops[-1] != start:
                stops.append(prev[stops[-1]])
            return Route(stops[::-1], round(d, 2), network.climb(stops[::-1]))
        for nxt, km in sorted(network.paths[node].items()):
            if frozenset({node, nxt}) not in banned and d + km < dist.get(nxt, float("inf")):
                dist[nxt], prev[nxt] = d + km, node
                heapq.heappush(heap, (d + km, nxt))
    return None

def paths_of(stops):
    return {frozenset(p) for p in zip(stops, stops[1:])}

def loop_via(network, start, turn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo\n\ndef test_loop():\n    \"out one way, back another\"\n    net = sample_geo()\n    loop = loop_via(net, \"Mill\", \"Farm\")\n    assert loop.stops == [\"Mill\", \"Weir\", \"Farm\", \"Wood\", \"Lake\", \"Mill\"] and loop.km == 11.0 and loop.climb_m == 110\n    assert loop_via(net, \"Mill\", \"Church\").stops == [\"Mill\", \"Weir\", \"Church\", \"Mill\"]\n    assert loop_via(net, \"Mill\", \"Mill\") is None\n    net.add_path(\"Island\", \"Mill\", 5.0)\n    assert loop_via(net, \"Mill\", \"Island\") is None, \"a dead end has no way back\"")

### Exercise: Where to turn

Write `target_km(minutes)`: the flat distance Naismith's rule walks in that time (`minutes / 12`), rounded to 2 places. Then `turning_points(dist, target, low=0.3, high=0.6)`: the junctions (not the start, which is at distance 0) whose distance is between `low × target` and `high × target` inclusive, sorted by how close their distance is to `target / 2` (ties by name).

In [ ]:
def target_km(minutes):
    ___

def turning_points(dist, target, low=0.3, high=0.6):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_candidates():\n    \"about halfway\"\n    assert target_km(120) == 10.0 and target_km(95) == 7.92\n    dist = {\"Mill\": 0, \"Weir\": 1.5, \"Church\": 3.5, \"Farm\": 4.0, \"Lake\": 3.5, \"Ridge\": 6.0, \"Wood\": 5.5}\n    assert turning_points(dist, 10.0) == [\"Wood\", \"Farm\", \"Ridge\", \"Church\", \"Lake\"]\n    assert turning_points(dist, 4.0) == [\"Weir\"]\n    assert turning_points(dist, 100.0) == []")

### Bug hunt: A route walked backwards

Following `prev` from the end gives the stops in reverse, and the code forgets to turn them round: it prints `['Ridge', 'Farm', 'Weir', 'Mill']`. Reverse them, so it prints `['Mill', 'Weir', 'Farm', 'Ridge']`.

In [ ]:
prev = {"Weir": "Mill", "Farm": "Weir", "Ridge": "Farm"}
stops = ["Ridge"]
while stops[-1] != "Mill":
    stops.append(prev[stops[-1]])
print(stops)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"start first\"\n    expect_output(\"['Mill', 'Weir', 'Farm', 'Ridge']\")")

### Bug hunt: Back the way we came

The banned set holds the outward steps as ordered pairs, so walking the same path the other way isn't banned and the "round" walk retraces it: it prints `True`. Ban paths in both directions (use `frozenset` pairs), so it prints `False`.

In [ ]:
out = ["Mill", "Weir", "Farm"]
banned = set(zip(out, out[1:]))
back_step = ("Farm", "Weir")
print(back_step not in banned)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a path is the same both ways\"\n    expect_output(\"False\")")

### Puzzle: Round walks in a few lines

Arrange the core of the round-walk search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # how many directions a frozenset({"A", "B"}) ban covers
q = ___   # the flat distance, in km, Naismith's rule walks in 90 minutes
r = ___   # True or False: every junction on a loop is a turning point that produces the same loop
s = ___   # "exponential" or "polynomial": the number of simple loops as a map grows
t = ___   # junctions in prev for a connected map of 50 junctions searched from one of them

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 7.5 and r is False, \"p, q and r\"\n    assert s == \"exponential\" and t == 49, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: round walks, properly

Write `round_walks(network, start, minutes, tolerance=15)` using the given helpers: compute the shortest-path tree from `start` (raise `UnknownPlace` for an unknown start), take the turning points for `target_km(minutes)`, build a loop through each, skip `None`s and duplicates (the same set of paths), and keep loops within `tolerance` minutes of the target. Return a list of dicts `{"turn", "stops", "km", "minutes"}` (km rounded to 1 place) sorted by how far the minutes are from the target, then km, then stops. Also count the searches made: 1 for the tree, plus 2 (out and back) for each turning point tried. Return `(walks, searches)`.

In [ ]:
import heapq
from waymark_domain import Route, UnknownPlace
from waymark_geo import sample_geo, grid_network

def shortest_tree(network, start):
    dist, prev, done, heap = {start: 0}, {}, set(), [(0, start)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        for nxt, km in sorted(network.paths[node].items()):
            if d + km < dist.get(nxt, float("inf")):
                dist[nxt], prev[nxt] = d + km, node
                heapq.heappush(heap, (d + km, nxt))
    return {k: round(v, 2) for k, v in dist.items()}, prev

def shortest_avoiding(network, start, end, banned):
    dist, prev, done, heap = {start: 0}, {}, set(), [(0, start)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        if node == end:
            stops = [end]
            while stops[-1] != start:
                stops.append(prev[stops[-1]])
            return Route(stops[::-1], round(d, 2), network.climb(stops[::-1]))
        for nxt, km in sorted(network.paths[node].items()):
            if frozenset({node, nxt}) not in banned and d + km < dist.get(nxt, float("inf")):
                dist[nxt], prev[nxt] = d + km, node
                heapq.heappush(heap, (d + km, nxt))
    return None

def paths_of(stops):
    return {frozenset(p) for p in zip(stops, stops[1:])}

def target_km(minutes):
    return round(minutes / 12, 2)

def turning_points(dist, target, low=0.3, high=0.6):
    chosen = [n for n, d in dist.items() if d > 0 and low * target <= d <= high * target]
    return sorted(chosen, key=lambda n: (abs(dist[n] - target / 2), n))

def round_walks(network, start, minutes, tolerance=15):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo, grid_network\n\ndef test_sample():\n    \"good loops, no duplicates\"\n    walks, searches = round_walks(sample_geo(), \"Mill\", 150)\n    assert [(w[\"turn\"], w[\"stops\"], w[\"km\"], w[\"minutes\"]) for w in walks] == [\n        (\"Wood\", [\"Mill\", \"Lake\", \"Wood\", \"Farm\", \"Weir\", \"Mill\"], 11.0, 143)]\n    assert searches == 1 + 2 * 3, \"a tree, then out and back for each of three turning points\"\n\ndef test_big_map():\n    \"fast on a real-sized map\"\n    walks, searches = round_walks(grid_network(40), \"20-20\", 120, tolerance=10)\n    assert walks and all(abs(w[\"minutes\"] - 120) <= 10 for w in walks)\n    assert len({frozenset(zip(w[\"stops\"], w[\"stops\"][1:])) for w in walks}) == len(walks)\n    assert searches < 200\n    try:\n        round_walks(sample_geo(), \"Moon\", 60)\n    except UnknownPlace:\n        pass\n    else:\n        raise AssertionError(\"unknown start\")")

## Warm-up

Each extra junction allowed multiplies the number of loops (each one counted in both directions). On a map of thousands of junctions, a walk of two hours passes through dozens of them: no computer can try them all. A different approach is needed, not a faster computer.

## Why the spike failed

Week 87's challenge found **every** loop and kept the ones near the target time. That's exact, and exponential: the work doubles and redoubles with every junction a walk can pass through. The ADR chose a different idea, common in route planners: a round walk is an **outward route** to a **turning point** and a **return route** home. Pick the turning point well (about half the target distance away), take the shortest route there, and the shortest route back that avoids the outward paths. It won't find every possible loop, but it finds good ones, fast, and the work grows only with the number of candidate turning points.

## Distances to everywhere

Dijkstra's algorithm doesn't have to stop at one destination: run it until the heap is empty and it gives the shortest distance from the start to **every** reachable junction, plus, for each, the junction before it on that shortest route (`prev`). That's the **shortest-path tree** from the start. One run answers "how far is every junction?" (to choose turning points) and "what's the route to any of them?" (follow `prev` back from the junction to the start, then reverse).

## Out and back a different way

The return route must not walk back along the outward paths, or the "round" walk is a there-and-back. Search for it with those paths **banned**: the same Dijkstra, skipping any step along a banned path (in **either** direction, since a path walked back is the same path). Pass the banned paths in as a set of `frozenset({a, b})` pairs, rather than deleting them from the network: the network is shared (cached, even) and must not change.

If there's no way back without retracing (the turning point is at the end of a dead-end path), that turning point gives no round walk: skip it.

## Choosing turning points

A target of 2 hours is about 10 km on the flat (Naismith: 12 minutes per km), so a loop's outward half is roughly 5 km. Candidate turning points are junctions whose shortest distance from the start is between about **30% and 60%** of the target distance (the return route is usually longer than the outward one, because it avoids the obvious paths). For each candidate, build the loop, compute its time with the climb, and keep those within the tolerance. Different turning points can produce the **same** loop (any junction on a loop is its own turning point), so deduplicate by the set of paths used.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Your exponential spot**: is there any feature in your capstone that tries every combination, every order or every subset? Measure it at three sizes and see how fast it grows.
2. **Build today's story**, choosing a good-enough approach where an exact one doesn't scale, and record the trade-off in an ADR.
3. **A test on a realistic size**, with a work budget (searches, queries or loop iterations) rather than a time limit.
4. **Merge** and update the burndown.

## Before you go

Tomorrow is S9: walking preferences, so "the best route" can mean the flattest, the quietest or the shortest, depending on who's walking.